# Brain Age Batch Inference - Colab GPU

Runs **SFCN**, **SynthBA**, and **MIDIBrainAge** on IXI scans.

**Before running:** Runtime -> Change runtime type -> **GPU (T4)**

| Model | Paper | Preprocessing |
|-------|-------|---------------|
| SFCN | Peng et al. 2021 | N4 + deepbet + MNI affine + crop [160,192,160] |
| SynthBA | Puglisi et al. 2024 | built-in (SynthSeg + ANTs MNI) |
| MIDI | Wood et al. 2024 | built-in (HD-BET + ANTs MNI) |

## 0. Mount Drive and clone repo

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [8]:
import os

REPO = "/content/faceage-to-brainage"

if not os.path.exists(REPO):
    os.system(f"git clone https://github.com/kondratevakate/faceage-to-brainage.git {REPO}")
else:
    os.system(f"git -C {REPO} pull --rebase")

os.chdir(REPO)
print("Working dir:", os.getcwd())

Working dir: /content/faceage-to-brainage


## 1. Install dependencies

In [7]:
import os
from pathlib import Path

os.system('pip install -q numpy scipy pandas matplotlib tqdm Pillow nibabel '
          'scikit-image torch torchvision SimpleITK nilearn deepbet synthba '
          'antspyx monai pingouin seaborn')

if not os.path.exists('vendor/SFCN/brain_age'):
    os.system('git clone --depth 1 https://github.com/ha-ha-ha-han/UKBiobank_deep_pretrain vendor/SFCN')

# MIDIBrainAge is a git submodule — check for actual files, not just the dir
if not os.path.exists('vendor/MIDIBrainAge/run_inference.py'):
    os.system('git clone --depth 1 https://github.com/MIDIconsortium/BrainAge vendor/MIDIBrainAge')

# Patch pre_process.py for MONAI >= 1.0 (AddChannel was removed)
pp = Path('vendor/MIDIBrainAge/pre_process.py')
src = pp.read_text()
if 'AddChannel' in src and 'class AddChannel' not in src:
    out = []
    for line in src.splitlines():
        if line.strip().startswith('from monai.transforms') and 'AddChannel' in line:
            out += [
                'try:',
                '    from monai.transforms import AddChannel',
                'except ImportError:',
                '    class AddChannel:',
                '        def __call__(self, x):',
                '            import numpy as np',
                '            return np.expand_dims(x, 0)',
            ]
        else:
            out.append(line)
    pp.write_text(chr(10).join(out))
    print('MIDI pre_process.py patched for MONAI >= 1.0')

print('Install complete')

MIDI pre_process.py patched for MONAI >= 1.0
Install complete


## 2. Verify SFCN weights

The weights ship inside the cloned SFCN repo — no separate download needed.

In [9]:
import os

SFCN_WEIGHT = "vendor/SFCN/brain_age/run_20190719_00_epoch_best_mae.p"

if os.path.exists(SFCN_WEIGHT):
    print("OK:", SFCN_WEIGHT)
else:
    raise FileNotFoundError(
        f"SFCN weights not found at {SFCN_WEIGHT}. "
        "The file should have been cloned with the SFCN repo in step 1. "
        "Re-run the install cell."
    )

OK: vendor/SFCN/brain_age/run_20190719_00_epoch_best_mae.p


## 2.5. Find your data in Drive

In [13]:
import os
from pathlib import Path

DRIVE = '/content/drive/MyDrive'

# Show top-level Drive folders
print('=== Drive root ===');
for p in sorted(Path(DRIVE).iterdir()):
    print(' ', p.name, '/' if p.is_dir() else '')

# Search for CSV files that look like manifests
print(chr(10) + '=== CSV files in Drive (first 20) ===')
csvs = list(Path(DRIVE).rglob('*.csv'))[:20]
for p in csvs:
    print(' ', p)

# Search for NIfTI files
print(chr(10) + '=== NIfTI files in Drive (first 5) ===')
niis = list(Path(DRIVE).rglob('*.nii.gz'))[:5]
for p in niis:
    print(' ', p)

=== Drive root ===
  Atelic /
  Atelic Pharmacy-embedded Attendance Engine for Brain Health Referrals.gslides 
  Colab Notebooks /
  Connected facilities and EMRs.gsheet 
  Europass /
  HSE_PhD_thesis /
  HSIL Harvard Hackathon — All Winners & Participants 2021-2026.gsheet 
  InstituteDesign /
  Job search 2025 /
  LiteBC /
  MIDL 2026, rebuttal.gdoc 
  MISC /
  Maastricht University /
  Medical AI /
  Medim /
  Mentorship Fellowship.gsheet 
  Oncology AI.gsheet 
  One Pager Template.gslides 
  Overleaf 2022 /
  PhD thesis Belavin /
  PreCheck-AI-Powered-Pre-Visit-Layer-for-Solo-Practitioners.pdf 
  Saved from Chrome /
  Skoltech PhD thesis /
  WHX Dubai 2026 - Exhibitor List.gsheet 
  WHX Dubai 2026 - Speakers & LinkedIn Profiles.gdoc 
  brainage_preproc /
  brainage_results /
  docs /
  health.tech Summit 2026 — Partners & Companies.gsheet 
  healthcare_pm_v4_updated.xlsx 
  hellomri 
  hellomri (1) /
  history_taste /
  lead database.gsheet 
  pharma_pilot_leads.gsheet 
  pharmacy.d

## 3. Configure paths

**Edit the variables below** to match your Drive layout.

In [10]:
import json, os
from pathlib import Path
import pandas as pd

REPO = '/content/faceage-to-brainage'

# Paste the path you found in the diagnostic cell above
# Shared folders added as shortcut appear under MyDrive/FolderName
# Shared drives appear under /content/drive/Shareddrives/DriveName
IXI_MANIFEST = '/content/drive/MyDrive/IXI/ixi_t1_manifest.csv'
RESULTS_DIR  = '/content/drive/MyDrive/brainage_results'
PREPROC_DIR  = '/content/drive/MyDrive/brainage_preproc'

# Sanity check
if not Path(IXI_MANIFEST).exists():
    # Try to find it automatically
    candidates = list(Path('/content/drive').rglob('ixi_t1_manifest.csv'))
    if candidates:
        IXI_MANIFEST = str(candidates[0])
        print('Auto-found manifest:', IXI_MANIFEST)
    else:
        raise FileNotFoundError(
            'ixi_t1_manifest.csv not found in Drive. '
            'Add the shared folder as a shortcut to your Drive first.'
        )

df = pd.read_csv(IXI_MANIFEST)
print(f'Manifest OK: {len(df)} rows')
print(df.head(3).to_string())

os.makedirs(f'{RESULTS_DIR}/ixi', exist_ok=True)
os.makedirs(f'{PREPROC_DIR}/ixi', exist_ok=True)

config = {
    'runtime': {'device': 'cuda'},
    'sfcn': {
        'model_dir':          f'{REPO}/vendor/SFCN',
        'weight_path':        f'{REPO}/vendor/SFCN/brain_age/run_20190719_00_epoch_best_mae.p',
        'skullstrip_command': 'deepbet',
        'skip_skullstrip':    False,
        'keep_skullstripped': False,
        'n4_correct':         True,
        'register_mni':       True,
        'age_bins':           {'start': 42.0, 'step': 1.0, 'count': 40}
    },
    'datasets': {
        'ixi': {
            'manifest_csv':      IXI_MANIFEST,
            'input_path_column': 't1_filename',
            'chron_age_column':  'AGE',
            'subject_id_column': 'IXI_ID',
            'output_csv':        f'{RESULTS_DIR}/ixi/predictions.csv',
            'preproc_dir':       f'{PREPROC_DIR}/ixi'
        }
    }
}

cfg_path = f'{REPO}/config/local/brain_age_local.json'
os.makedirs(os.path.dirname(cfg_path), exist_ok=True)
Path(cfg_path).write_text(json.dumps(config, indent=2))
print('Config written:', cfg_path)

Config written: /content/faceage-to-brainage/config/local/brain_age_local.json


## 4. Smoke test - 3 scans, all models

In [12]:
import subprocess, sys

def run_batch(model, tta=False, limit=3, dataset='ixi'):
    cmd = [
        sys.executable, 'scripts/batch_brainage.py',
        '--model', model,
        '--dataset', dataset,
        '--config', 'config/local/brain_age_local.json',
    ]
    if limit is not None:
        cmd += ['--limit', str(limit)]
    if tta:
        cmd.append('--tta')
    print('>>> Running:', ' '.join(cmd))
    r = subprocess.run(cmd, text=True, capture_output=True)
    if r.stdout:
        print(r.stdout)
    if r.returncode != 0:
        print(f'FAILED (exit {r.returncode})')
        if r.stderr:
            print('STDERR:', r.stderr[-3000:])
    return r.returncode

run_batch('sfcn', limit=3)

>>> Running: /usr/bin/python3 scripts/batch_brainage.py --model sfcn --dataset ixi --config config/local/brain_age_local.json --limit 3
FAILED (exit 1)
STDERR: Traceback (most recent call last):
  File "/content/faceage-to-brainage/scripts/batch_brainage.py", line 209, in <module>
    main()
  File "/content/faceage-to-brainage/scripts/batch_brainage.py", line 121, in main
    frame = pd.read_csv(manifest_csv)
            ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1620, in __init__
    self._engine = self._make_engine(f, self.eng

1

In [23]:
run_batch("synthba", limit=3)

>>> Running: /usr/bin/python3 scripts/batch_brainage.py --model synthba --dataset ixi --config config/local/brain_age_local.json --limit 3


1

In [24]:
run_batch("midi", limit=3)

>>> Running: /usr/bin/python3 scripts/batch_brainage.py --model midi --dataset ixi --config config/local/brain_age_local.json --limit 3


1

## 5. Check smoke test results

In [ ]:
import pandas as pd
from pathlib import Path

for csv in sorted(Path(RESULTS_DIR, "ixi").glob("*.csv")):
    df = pd.read_csv(csv)
    ok = (df["status"] == "ok").sum()
    print(f"{csv.name}  ({ok}/{len(df)} ok)")
    print(df[["subject_id","chron_age","predicted_age","brain_age_gap","status"]].to_string())
    print()

## 6. Full batch - all IXI scans

Each model writes its own CSV. GPU runtime: ~30 min total for 561 scans.

In [ ]:
run_batch("sfcn", limit=None)

In [ ]:
run_batch("synthba", limit=None)

In [ ]:
run_batch("midi", limit=None)

## 7. Compare models

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

frames = [pd.read_csv(p) for p in Path(RESULTS_DIR, "ixi").glob("*.csv")]
all_df = pd.concat(frames, ignore_index=True)
ok = all_df[all_df["status"] == "ok"].copy()

stats = ok.groupby("model_name").apply(lambda g: pd.Series({
    "n":    len(g),
    "MAE":  g["brain_age_gap"].abs().mean(),
    "bias": g["brain_age_gap"].mean(),
    "r":    g[["chron_age","predicted_age"]].corr().iloc[0,1]
})).round(2)
print(stats)

models = ok["model_name"].unique()
fig, axes = plt.subplots(1, len(models), figsize=(6*len(models), 5))
if len(models) == 1:
    axes = [axes]
for ax, (model, gdf) in zip(axes, ok.groupby("model_name")):
    ax.scatter(gdf["chron_age"], gdf["predicted_age"], alpha=0.4, s=10)
    lims = [gdf["chron_age"].min()-2, gdf["chron_age"].max()+2]
    ax.plot(lims, lims, "k--", lw=0.8)
    mae = gdf["brain_age_gap"].abs().mean()
    ax.set_title(f"{model} / MAE={mae:.1f} y")
    ax.set_xlabel("Chronological age"); ax.set_ylabel("Predicted age")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/scatter_all_models.png", dpi=150)
plt.show()